# 06 - Modelo Local e Remoto

Notebook híbrido para gerar SQL com um modelo local no `Ollama` e usar `OpenRouter` apenas para redigir a resposta final.

## Objetivo
- Traduzir perguntas em linguagem natural para SQL usando `Qwen2.5-Coder` local.
- Executar consultas em `SQLite` localmente.
- Usar `Claude Sonnet 4.6` no OpenRouter apenas para a resposta final.

## Ordem de execução

1. Garanta que o `Ollama` esteja instalado e com o serviço iniciado: `ollama serve`.
2. Baixe o modelo local para geração de SQL: `ollama pull qwen2.5-coder:7b-instruct`.
3. Defina a variável de ambiente `OPENROUTER_API_KEY`.
4. Execute a célula de setup do banco e do dicionário de dados.
5. Execute a célula de configuração dos modelos e do estado do agente.
6. Execute a célula com os nós do fluxo e compile o grafo.
7. Execute a célula final para rodar o teste de estresse e gerar o relatório.

Observação: se a sua CPU ficar muito lenta com `qwen2.5-coder:7b-instruct`, troque `LOCAL_SQL_MODEL` para `qwen2.5-coder:3b-instruct`.

### Célula 1: Setup do banco e contexto técnico

In [1]:
import os
import sqlite3
import time
from typing import Any, Dict, List

import ollama
import pandas as pd
from openai import OpenAI
from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict


DB_NAME = "oil.db"
DB_PATH = os.path.abspath(DB_NAME)

if "conn" in globals():
    try:
        conn.close()
    except Exception:
        pass

if os.path.exists(DB_PATH):
    try:
        os.remove(DB_PATH)
    except OSError as exc:
        print(f"[Aviso] Banco antigo não pôde ser removido: {exc}")

setup_conn = sqlite3.connect(DB_PATH)
setup_cursor = setup_conn.cursor()
setup_cursor.execute(
    """
    CREATE TABLE well_production (
        well_name TEXT,
        field_name TEXT,
        production_date TEXT,
        oil_bbl REAL,
        gas_mscf REAL,
        water_bbl REAL,
        hours_on REAL
    )
    """
)

rows = [
    ("WELL-A1", "FIELD-X", "2026-06-01", 1200, 800, 300, 24),
    ("WELL-A2", "FIELD-X", "2026-06-01", 900, 600, 500, 24),
    ("WELL-B1", "FIELD-Y", "2026-06-01", 1500, 1100, 200, 24),
    ("WELL-A1", "FIELD-X", "2026-06-02", 1250, 820, 320, 24),
    ("WELL-A2", "FIELD-X", "2026-06-02", 920, 620, 510, 24),
    ("WELL-B1", "FIELD-Y", "2026-06-02", 1520, 1120, 210, 24),
    ("WELL-A1", "FIELD-X", "2026-06-03", 1230, 810, 310, 24),
    ("WELL-A2", "FIELD-X", "2026-06-03", 910, 610, 505, 24),
    ("WELL-B1", "FIELD-Y", "2026-06-03", 1550, 1150, 220, 24),
]

setup_cursor.executemany(
    "INSERT INTO well_production VALUES (?,?,?,?,?,?,?)",
    rows,
)
setup_conn.commit()

schema_df = pd.read_sql("PRAGMA table_info(well_production)", setup_conn)
schema_text = ", ".join(
    f"{row['name']} ({row['type']})"
    for _, row in schema_df.iterrows()
)
setup_conn.close()

conn = sqlite3.connect(
    f"file:{DB_PATH}?mode=ro",
    uri=True,
    check_same_thread=False,
)

DATA_DICTIONARY = """
TABELA: well_production
CONTEXTO: Histórico diário de produção física de fluidos por poço petrolífero.

COLUNAS:
- well_name (TEXT): Nome identificador único do poço (Ex: WELL-A1, WELL-B1).
- field_name (TEXT): Nome do campo de produção onde o poço está alocado (Ex: FIELD-X, FIELD-Y).
- production_date (TEXT): Data da medição do volume no formato ISO YYYY-MM-DD.
- oil_bbl (REAL): Volume diário de óleo produzido medido em barris (bbl).
- gas_mscf (REAL): Volume diário de gás produzido medido em milhares de pés cúbicos standard (mscf).
- water_bbl (REAL): Volume diário de água produzida medido em barris (bbl).
- hours_on (REAL): Total de horas em que o poço permaneceu aberto e produzindo no dia (máximo 24.0).
""".strip()

print("[GOVERNANÇA] Banco de dados inicializado em modo read-only.")
schema_df

[GOVERNANÇA] Banco de dados inicializado em modo read-only.


,cid,name,type,notnull,dflt_value,pk
0,0,well_name,TEXT,0,None,0
1,1,field_name,TEXT,0,None,0
2,2,production_date,TEXT,0,None,0
3,3,oil_bbl,REAL,0,None,0
4,4,gas_mscf,REAL,0,None,0
5,5,water_bbl,REAL,0,None,0
6,6,hours_on,REAL,0,None,0


### Célula 2: Configuração do modelo e estado do agente

In [2]:
LOCAL_SQL_MODEL = os.environ.get("LOCAL_SQL_MODEL", "qwen2.5-coder:7b-instruct")
REMOTE_RESPONSE_MODEL = "anthropic/claude-sonnet-4.6"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_HEADERS = {
    "HTTP-Referer": "http://localhost:3000",
    "X-Title": "Sql Oil Assistant",
}


class AgentState(TypedDict):
    question: str
    generated_sql: str
    error_message: str
    retry_count: int
    query_result: str
    final_response: str



def safe_str(text: str | None) -> str:
    if text is None:
        return ""
    return str(text).encode("utf-8", errors="ignore").decode("utf-8")



def format_llm_error(exc: Exception) -> str:
    parts = [f"{type(exc).__name__}: {safe_str(str(exc))}"]

    status_code = getattr(exc, "status_code", None)
    if status_code is not None:
        parts.append(f"status={status_code}")

    response = getattr(exc, "response", None)
    if response is not None:
        response_text = safe_str(getattr(response, "text", ""))[:400]
        if response_text:
            parts.append(f"body={response_text}")

    return " | ".join(part for part in parts if part)



def format_local_error(exc: Exception) -> str:
    if isinstance(exc, ollama.RequestError):
        return (
            f"Ollama indisponível em {OLLAMA_HOST}. "
            f"Inicie com `ollama serve`. Detalhe: {safe_str(str(exc))}"
        )

    if isinstance(exc, ollama.ResponseError):
        if getattr(exc, "status_code", None) == 404:
            return (
                f"Modelo local '{LOCAL_SQL_MODEL}' não encontrado. "
                f"Baixe com `ollama pull {LOCAL_SQL_MODEL}`."
            )
        return f"ResponseError(status={exc.status_code}): {safe_str(str(exc))}"

    return f"{type(exc).__name__}: {safe_str(str(exc))}"



def extract_text_content(content: Any) -> str:
    if isinstance(content, str):
        return safe_str(content)

    if isinstance(content, list):
        parts: List[str] = []
        for item in content:
            if isinstance(item, str):
                parts.append(safe_str(item))
                continue

            if isinstance(item, dict) and item.get("type") == "text":
                text = item.get("text", "")
                if text:
                    parts.append(safe_str(text))

        return "\n".join(part for part in parts if part).strip()

    return safe_str(content)


if not OPENROUTER_API_KEY:
    print("[Aviso] OPENROUTER_API_KEY não encontrada no ambiente.")

ollama_client = ollama.Client(host=OLLAMA_HOST)
openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY or "missing_api_key",
    base_url=OPENROUTER_BASE_URL,
)

LOCAL_SQL_MODEL_READY = False
LOCAL_SQL_SYSTEM_PROMPT = (
    "Você é um engenheiro de software especialista em banco de dados, "
    "com especialidade em SQLite. "
    "Sua função neste fluxo é traduzir perguntas em linguagem natural "
    "para consultas SQL corretas, seguras e compatíveis com SQLite. "
    "Use apenas o esquema fornecido e retorne somente SQL puro quando solicitado. "
    "Jamais retorne nenhum tipo de formatação, como por exemplo markdown. "
)



def probe_local_sql_runtime() -> str:
    try:
        ollama_client.show(LOCAL_SQL_MODEL)
        return f"[LOCAL SQL] Modelo disponível no Ollama: {LOCAL_SQL_MODEL}"
    except Exception as exc:
        return f"[LOCAL SQL] {format_local_error(exc)}"



def ensure_local_sql_model_ready() -> None:
    global LOCAL_SQL_MODEL_READY
    if LOCAL_SQL_MODEL_READY:
        return

    try:
        ollama_client.show(LOCAL_SQL_MODEL)
        LOCAL_SQL_MODEL_READY = True
    except Exception as exc:
        raise RuntimeError(format_local_error(exc)) from exc



def invoke_local_sql_model(prompt: str) -> str:
    ensure_local_sql_model_ready()

    response = ollama_client.chat(
        model=LOCAL_SQL_MODEL,
        messages=[
            {"role": "system", "content": LOCAL_SQL_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        options={
            "temperature": 0,
            "num_predict": 220,
            "num_ctx": 8192,
        },
        keep_alive="30m",
    )

    content = extract_text_content(response.message.content)
    if not content:
        raise ValueError("O modelo local retornou conteúdo vazio.")

    return content



def invoke_openrouter(prompt: str) -> str:
    if not OPENROUTER_API_KEY:
        raise RuntimeError("OPENROUTER_API_KEY não encontrada no ambiente.")

    response = openrouter_client.chat.completions.create(
        model=REMOTE_RESPONSE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        extra_headers=OPENROUTER_HEADERS,
    )

    if not response.choices:
        raise ValueError("OpenRouter retornou uma resposta sem choices.")

    message = response.choices[0].message
    content = extract_text_content(message.content)
    if not content:
        raise ValueError("OpenRouter retornou conteúdo vazio.")

    return content


print(probe_local_sql_runtime())
print(f"[REMOTE RESPONSE] Modelo configurado: {REMOTE_RESPONSE_MODEL}")

[LOCAL SQL] Modelo disponível no Ollama: qwen2.5-coder:7b-instruct
[REMOTE RESPONSE] Modelo configurado: anthropic/claude-sonnet-4.6


### Célula 3: Nós do fluxo e compilação do LangGraph

In [3]:
from textwrap import dedent

PROHIBITED_KEYWORDS = {
    "DROP",
    "DELETE",
    "INSERT",
    "UPDATE",
    "ALTER",
    "CREATE",
    "TRUNCATE",
    "EXECUTE"
}



def generate_sql_node(state: AgentState) -> Dict[str, Any]:
    error_message = state.get("error_message", "")
    error_context = ""
    if error_message:
        error_context = (
            "\nATENÇÃO: sua tentativa anterior falhou com o erro: "
            f"{error_message}. Corrija a sintaxe."
        )

    prompt = dedent(
        f"""
        Sua tarefa é gerar SQL SQLite para uma tabela de produção de petróleo.

        Esquema:
        {schema_text}

        Dicionário de dados:
        {DATA_DICTIONARY}

        Regras:
        1. Retorne somente SQL puro, sem markdown.
        2. Use apenas SELECT.
        3. Use LIMIT em vez de TOP.
        4. Para agregações por poço ou campo, use GROUP BY.
        5. Traga a métrica junto com o nome do poço ou campo.
        6. Convenções:
           - GOR = gas_mscf / oil_bbl
           - Water Cut = water_bbl / (oil_bbl + water_bbl)
        {error_context}

        Pergunta: {state["question"]}
        SQL:
        """
    ).strip()

    try:
        clean_sql = (
            invoke_local_sql_model(safe_str(prompt))
            .replace("```sql", "")
            .replace("```", "")
            .replace(";", "")
            .strip()
        )
        return {
            "generated_sql": safe_str(clean_sql),
            "retry_count": state.get("retry_count", 0) + 1,
        }
    except Exception as exc:
        return {
            "error_message": f"Erro no modelo local de SQL: {safe_str(str(exc))}",
            "retry_count": state.get("retry_count", 0) + 1,
        }



def execute_sql_node(state: AgentState) -> Dict[str, Any]:
    error_message = state.get("error_message", "")
    if "Erro no modelo local de SQL" in error_message:
        return {"query_result": ""}

    generated_sql = state.get("generated_sql", "").strip()
    if not generated_sql:
        return {
            "error_message": "Nenhum SQL válido foi gerado.",
            "query_result": "",
        }

    sql_to_run = generated_sql.upper()
    if any(keyword in sql_to_run for keyword in PROHIBITED_KEYWORDS):
        return {
            "error_message": "Bloqueio de segurança: comando de escrita proibido.",
            "query_result": "",
        }

    try:
        df = pd.read_sql(generated_sql, conn)
        return {
            "query_result": safe_str(df.to_string(index=False)),
            "error_message": "",
        }
    except Exception as exc:
        return {
            "error_message": safe_str(str(exc)),
            "query_result": "",
        }



def respond_node(state: AgentState) -> Dict[str, Any]:
    error_message = state.get("error_message", "")
    if error_message:
        return {
            "final_response": (
                "Não foi possível responder devido a um erro persistente: "
                f"{error_message}"
            )
        }

    prompt = dedent(
        f"""
        Você é um engenheiro sênior de produção de petróleo e gás natural.

        Responda usando somente os dados retornados abaixo.

        Regras:
        1. Não invente ativos, datas, comparações ou explicações não presentes nos dados.
        2. Não afirme que um ativo é o único da base se a consulta retornou apenas o primeiro colocado.
        3. Cite os nomes dos ativos e os valores exatos retornados.
        4. Seja curto, técnico e conclusivo.

        Pergunta: {state["question"]}
        Dados do banco:
        {state["query_result"]}
        Resposta técnico-comercial:
        """
    ).strip()

    try:
        final_response = invoke_openrouter(safe_str(prompt))
        return {"final_response": safe_str(final_response.strip())}
    except Exception as exc:
        return {
            "final_response": (
                "Erro na geração da resposta final: "
                f"{format_llm_error(exc)}"
            )
        }



def should_retry_or_respond(state: AgentState) -> str:
    if state.get("error_message") and state.get("retry_count", 0) < 3:
        return "generate_sql"
    return "respond"


workflow = StateGraph(AgentState)
workflow.add_node("generate_sql", generate_sql_node)
workflow.add_node("execute_sql", execute_sql_node)
workflow.add_node("respond", respond_node)

workflow.add_edge(START, "generate_sql")
workflow.add_edge("generate_sql", "execute_sql")
workflow.add_conditional_edges(
    "execute_sql",
    should_retry_or_respond,
    {"generate_sql": "generate_sql", "respond": "respond"},
)
workflow.add_edge("respond", END)

app = workflow.compile()
print("[DIAGNÓSTICO] Grafo híbrido compilado: SQL local + resposta final remota.")

[DIAGNÓSTICO] Grafo híbrido compilado: SQL local + resposta final remota.


### Célula 4: Teste de estresse e geração do relatório

In [4]:
test_questions = [
    "Qual poço teve maior produção acumulada de óleo?",
    "Qual poço produziu mais água?",
    "Qual poço produziu mais gás?",
    "Qual campo produziu mais água?",
    "Qual poço teve menor produção de óleo?",
    "Qual poço teve maior produção média de óleo?",
]

report_dir = os.path.join(os.getcwd(), "notebooks")
if os.path.isdir(report_dir):
    REPORT_PATH = os.path.join(report_dir, "relatorio_modelo_local_e_remoto.md")
else:
    REPORT_PATH = os.path.abspath("relatorio_modelo_local_e_remoto.md")

results: List[Dict[str, Any]] = []
total_start_time = time.time()

print("=" * 80)
print(f"INICIANDO TESTE DE ESTRESSE COM {len(test_questions)} PERGUNTAS")
print("=" * 80)

for index, question in enumerate(test_questions, start=1):
    print(f"[TESTE {index}/{len(test_questions)}] {question}")

    initial_state: AgentState = {
        "question": safe_str(question),
        "generated_sql": "",
        "error_message": "",
        "retry_count": 0,
        "query_result": "",
        "final_response": "",
    }

    step_start_time = time.time()
    output = app.invoke(initial_state, {"recursion_limit": 15})
    elapsed = time.time() - step_start_time

    results.append(
        {
            "index": index,
            "question": question,
            "elapsed": elapsed,
            "retry_count": max(output.get("retry_count", 0) - 1, 0),
            "generated_sql": output.get("generated_sql", ""),
            "query_result": output.get("query_result", ""),
            "error_message": output.get("error_message", ""),
            "final_response": output.get("final_response", ""),
        }
    )

total_duration = time.time() - total_start_time
success_count = sum(1 for result in results if not result["error_message"])
failed_count = len(results) - success_count
overall_status = (
    "Concluído com sucesso"
    if failed_count == 0
    else f"Concluído com falhas ({failed_count}/{len(results)})"
)

with open(REPORT_PATH, "w", encoding="utf-8") as md:
    md.write("# Relatório Executivo - Modelo Local e Remoto\n\n")
    md.write(f"**Data da Execução:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    md.write(f"**Modelo Local (SQL):** {LOCAL_SQL_MODEL} via Ollama\n")
    md.write(f"**Modelo Remoto (Resposta Final):** {REMOTE_RESPONSE_MODEL} via OpenRouter\n\n")
    md.write("## 1. Dicionário de Dados Utilizado\n\n")
    md.write("```text\n")
    md.write(DATA_DICTIONARY)
    md.write("\n```\n\n")
    md.write("## 2. Histórico de Execuções e Respostas Técnicas\n\n")

    for result in results:
        case_index = result["index"]
        question_text = safe_str(result["question"])
        elapsed_text = "{:.2f}".format(result["elapsed"])
        retry_count = result["retry_count"]
        status_text = safe_str(result["error_message"]) or "Sem erros"
        sql_text = safe_str(result["generated_sql"])
        query_result_text = safe_str(result["query_result"])
        final_response_text = safe_str(result["final_response"])

        md.write(f"### Caso de Teste {case_index}: {question_text}\n")
        md.write(f"- Tempo de Resposta: {elapsed_text} segundos\n")
        md.write(f"- Tentativas de Correção (Retries): {retry_count}\n")
        md.write(f"- Status: {status_text}\n\n")
        md.write("```sql\n")
        md.write(f"{sql_text}\n")
        md.write("```\n\n")
        md.write("```text\n")
        md.write(f"{query_result_text}\n")
        md.write("```\n\n")
        md.write(f"> {final_response_text}\n\n")
        md.write("---\n\n")

    md.write("## 3. Sumário Executivo de Performance\n\n")
    md.write(f"- Total de Perguntas Submetidas: {len(results)}\n")
    md.write(f"- Casos com sucesso: {success_count}\n")
    md.write(f"- Casos com falha: {failed_count}\n")
    md.write(f"- Tempo Total de Varredura: {total_duration:.2f} segundos\n")
    md.write(f"- Média de Tempo por Requisição: {total_duration / len(results):.2f} segundos\n")
    md.write(f"- Status Geral do Sistema: {overall_status}\n")

print("=" * 80)
print(overall_status)
print(f"Relatório salvo em: {REPORT_PATH}")
print("=" * 80)

INICIANDO TESTE DE ESTRESSE COM 6 PERGUNTAS
[TESTE 1/6] Qual poço teve maior produção acumulada de óleo?
[TESTE 2/6] Qual poço produziu mais água?
[TESTE 3/6] Qual poço produziu mais gás?
[TESTE 4/6] Qual campo produziu mais água?
[TESTE 5/6] Qual poço teve menor produção de óleo?
[TESTE 6/6] Qual poço teve maior produção média de óleo?
Concluído com sucesso
Relatório salvo em: /home/wolf/Documentos/lab-artificial-inteligence/notebooks/relatorio_modelo_local_e_remoto.md
